# TerraWatch M4 | Learned pipeline demo

Use a GPU runtime (T4 or L4 for training, L4 for learned inference). Upload the companion `drive_members.py` when prompted, then sign in to the account with access to the extracted LEVIR-MCI folders. Run all cells and retain the results ZIP.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import shutil
import subprocess
import sys
import zipfile

import torch
from google.colab import files

assert torch.cuda.is_available(), 'Select an L4 GPU runtime'
gpu = torch.cuda.get_device_properties(0)
print(gpu.name, round(gpu.total_memory / 1024**3, 1), 'GiB')
assert gpu.total_memory >= 19 * 1024**3, 'Use an L4 for the unquantised referee'

helper = Path('/content/drive_members.py')
if not helper.is_file():
    print('Upload drive_members.py')
    uploaded = files.upload()
    if 'drive_members.py' not in uploaded:
        raise FileNotFoundError('Choose drive_members.py')
    helper.write_bytes(uploaded['drive_members.py'])

sys.path.insert(0, '/content')
from drive_members import DriveMembers

drive_data = DriveMembers.connect()

# If you already uploaded weights, enter their path here, e.g. /content/lr_1e-4.pt.
CHECKPOINT_OR_TRAINING_ZIP = ''

if CHECKPOINT_OR_TRAINING_ZIP:
    selected_input = Path(CHECKPOINT_OR_TRAINING_ZIP)
else:
    print('Upload lr_1e-4.pt or the Notebook 1 results ZIP')
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload exactly one .pt or results .zip file')
    filename, data = next(iter(uploaded.items()))
    selected_input = Path('/content') / Path(filename).name
    selected_input.write_bytes(data)

checkpoint = Path('/content/bit_m4.pt')

if selected_input.suffix.lower() == '.zip':
    with zipfile.ZipFile(selected_input) as archive:
        receipt = json.loads(archive.read('selection.json'))
        choice = receipt['selected']
        if not choice:
            raise ValueError('This ZIP has no selected checkpoint. Upload a .pt file.')
        name = choice['checkpoint']
        weights = archive.read(name)
        if hashlib.sha256(weights).hexdigest() != receipt['checkpoint_sha256'][name]:
            raise ValueError('Checkpoint hash mismatch')
        checkpoint.write_bytes(weights)
elif selected_input.suffix.lower() == '.pt':
    if selected_input.resolve() != checkpoint:
        shutil.copy2(selected_input, checkpoint)
else:
    raise ValueError('Choose a .pt checkpoint or the Notebook 1 results ZIP')

print('BIT checkpoint ready at', checkpoint)

## Prepare isolated worker environments and pinned assets

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os
import subprocess
import sys

repo = Path('/content/terrawatch-m4-demo')
if not repo.exists():
    subprocess.run([
        'git', 'clone', '--branch', 'milestone-4-review',
        'https://github.com/shoaib2000857/Group-7-DS-and-AI-Lab-Project.git',
        str(repo),
    ], check=True, timeout=180)

os.chdir(repo)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'opencv-python-headless', 'pydantic>=2,<3', 'einops',
], check=True, timeout=180)
subprocess.run(
    [sys.executable, 'scripts/setup_model_sources.py'],
    check=True, timeout=240,
)

# Create fresh worker environments with pip seeded by virtualenv.
bootstrap = '/content/tw_virtualenv_bootstrap'
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    '--target', bootstrap, 'virtualenv',
], check=True, timeout=180)

bootstrap_env = os.environ.copy()
bootstrap_env['PYTHONPATH'] = os.pathsep.join(
    p for p in [bootstrap, bootstrap_env.get('PYTHONPATH')] if p
)

caption_dir = '/content/tw-caption-env-seeded'
judge_dir = '/content/tw-referee-env-seeded'
caption_python = f'{caption_dir}/bin/python'
judge_python = f'{judge_dir}/bin/python'

for folder in [caption_dir, judge_dir]:
    subprocess.run([
        sys.executable, '-m', 'virtualenv', '--system-site-packages',
        '--seeder', 'app-data', '--no-download', folder,
    ], env=bootstrap_env, check=True, timeout=120)
    subprocess.run(
        [f'{folder}/bin/python', '-m', 'pip', '--version'],
        check=True, timeout=30,
    )

subprocess.run([
    caption_python, '-m', 'pip', 'install', '-q',
    'transformers==4.39.3', 'setuptools<70', 'wheel',
    'ftfy', 'regex', 'einops', 'tqdm',
], check=True, timeout=240)
subprocess.run([
    caption_python, '-m', 'pip', 'install', '-q', '--no-build-isolation',
    'git+https://github.com/openai/CLIP.git@dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1',
], check=True, timeout=180)
subprocess.run([
    judge_python, '-m', 'pip', 'install', '-q',
    'transformers==5.17.0', 'accelerate>=1.1,<2',
    'sentencepiece', 'safetensors',
], check=True, timeout=240)

output = Path('/content') / datetime.now(timezone.utc).strftime(
    'TerraWatch_M4_Learned_%Y%m%d_%H%M%S'
)
output.mkdir()

captions = output / 'LevirCCcaptions.json'
captions.write_bytes(
    drive_data.read('LEVIR-MCI-dataset/LevirCCcaptions.json')
)

weights = repo / 'models/checkpoints/rscama.pth'
if not weights.exists():
    subprocess.run([sys.executable, 'scripts/fetch_model_assets.py', '--rscama'],
                   check=True, timeout=900)
with weights.open('rb') as stream:
    digest = hashlib.file_digest(stream, 'sha256').hexdigest()
if digest != '5eeba271fb7c2ff4722b725cbf5ce1a1fdc0a33ddf1b077dcf5381aebd643449':
    raise ValueError('RSCaMa checkpoint checksum mismatch')
sys.path.insert(0, str(repo))
from src.training.captioner import upstream_tokenizer
tokenize, build_vocab, _ = upstream_tokenizer(repo / 'models/external/RSCaMa')
rows = json.loads(captions.read_text(encoding='utf-8'))['images']
vocabulary = build_vocab([(r['filename'], [tokenize(s['raw']) for s in r['sentences']]) for r in rows], 5)
if len(vocabulary) != 468:
    raise ValueError(f'Expected 468 vocabulary entries, got {len(vocabulary)}')
weights.with_name('rscama_vocab.json').write_text(json.dumps(vocabulary, indent=2), encoding='utf-8')
print('RSCaMa checkpoint and vocabulary ready.')


## Execute original T1/T2 pairs through the learned workers

In [ ]:
config=json.loads(Path('configs/pipeline.json').read_text())
config['detector'].update(python=sys.executable,checkpoint=str(checkpoint))
config['captioner']['python']=caption_python
config['referee']['python']=judge_python
config['worker_timeout_seconds']=420
config_path=output/'pipeline.json';config_path.write_text(json.dumps(config,indent=2))
subprocess.run([sys.executable,'scripts/check_model_setup.py','--config',str(config_path),'--environments','--strict'],check=True,timeout=150)
for pair in ('train_000010','val_000005'):
    inputs=output/'inputs'/pair;inputs.mkdir(parents=True)
    for date in ('T1','T2'):
        shutil.copy2(repo/'src/app/assets'/f'{pair}_{date}.png',inputs/f'{date}.png')
    case=output/pair
    try:
        subprocess.run([sys.executable,'scripts/run_pipeline.py','--t1',str(inputs/'T1.png'),'--t2',str(inputs/'T2.png'),'--pair-id',pair,'--config',str(config_path),'--output',str(case)],check=True,timeout=720)
        result=json.loads((case/'report.json').read_text());print(pair,result['generated_caption'],[(r['source_span'],r['decision']['outcome']) for r in result['claims']],result['errors'])
        if (case/'detector_response.mask.png').exists():
            for date in ('T1','T2'):shutil.copy2(inputs/f'{date}.png',case/f'{date}.png')
            (case/'review_inputs.json').write_text(json.dumps({'images':['T1.png','T2.png'],'mask':'detector_response.mask.png'}))
    except (subprocess.TimeoutExpired,subprocess.CalledProcessError) as error:
        (output/f'{pair}_execution_error.txt').write_text(str(error));print(pair,error)


## Export reports, masks, crops and worker logs

In [ ]:
captions.unlink(missing_ok=True)
archive=shutil.make_archive(str(output),'zip',output)
files.download(archive)
